# PV project

# Loading the selected data into database

## Install required libraries

In [ ]:
## Install the required libraries
%pip install duckdb pandas numpy scipy streamlit langchain-community

## Import Libraries

In [1]:
import duckdb
import os
from pathlib import Path
import pandas as pd
import numpy as np


## Identify the Folder Locations

In [4]:
print("Current Working Directory:", os.getcwd())
print("Contents of CWD:", os.listdir("."))


Current Working Directory: /Users/shymashameer/Desktop/Final Project/faers/notebooks
Contents of CWD: ['.DS_Store', '01_Loading Data.ipynb', '04_Forecasting.ipynb', '02_Initial EDA.ipynb', '.ipynb_checkpoints', '03_Exploration.ipynb']


In [5]:
# Check current working directory
cwd = Path.cwd()
print("Current Working Directory:", cwd)

# Walk up to 'faers' project root
project_root = cwd.parent if cwd.name == "notebooks" else cwd
print("Project Root:", project_root)

# List all directories and files under project_root
print("\n--- Project Tree ---")
for root, dirs, files in os.walk(project_root):
    # Skip hidden folders like .git, .ipynb_checkpoints
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    
    level = root.replace(str(project_root), '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        if not f.startswith('.'):
            print(f"{subindent}{f}")

Current Working Directory: /Users/shymashameer/Desktop/Final Project/faers/notebooks
Project Root: /Users/shymashameer/Desktop/Final Project/faers

--- Project Tree ---
faers/
    requirements.txt
    README.md
    data/
        faers_db.duckdb
        raw/
            faers_ascii_2015q1/
                FAQs.pdf
                Readme.pdf
                ascii/
                    drug15q1.pdf
                    REAC15Q1.txt
                    OUTC15Q1.txt
                    reac15q1.pdf
                    DRUG15Q1.txt
                    ASC_NTS.pdf
                    outc15q1.pdf
                    INDI15Q1.txt
                    RPSR15Q1.txt
                    ther15q1.pdf
                    DEMO15Q1.txt
                    rpsr15q1.pdf
                    THER15Q1.txt
                    indi15q1.pdf
                    demo15q1.pdf
            faers_ascii_2017q2/
                FAQs.pdf
                Readme.pdf
                ascii/
                    drug17q2.pdf
 

## Set the project paths

In [6]:
# Resolve project paths

# Set project root to parent directory of 'notebooks'
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

# Define absolute paths relative to project root
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
DB_PATH = PROJECT_ROOT / "data" / "faers_db.duckdb"

## Exploring the files in the folder

In [7]:

print("--- Searching for ALL files under project root ---")
found_files = []
for root, dirs, files in os.walk(project_root):
    for f in files:
        if not f.startswith('.'):  # Ignore hidden files
            full_path = os.path.join(root, f)
            found_files.append(full_path)

print(f"Total files found in project: {len(found_files)}\n")

# Filter and display all .txt or .pdf files found
data_files = [f for f in found_files if f.lower().endswith(('.txt', '.pdf'))]
print("--- Data / PDF Files Found ---")
for df in data_files[:20]:  # Show first 20
    print(df)

--- Searching for ALL files under project root ---
Total files found in project: 711

--- Data / PDF Files Found ---
/Users/shymashameer/Desktop/Final Project/faers/requirements.txt
/Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2015q1/FAQs.pdf
/Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2015q1/Readme.pdf
/Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2015q1/ascii/drug15q1.pdf
/Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2015q1/ascii/REAC15Q1.txt
/Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2015q1/ascii/OUTC15Q1.txt
/Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2015q1/ascii/reac15q1.pdf
/Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2015q1/ascii/DRUG15Q1.txt
/Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2015q1/ascii/ASC_NTS.pdf
/Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2015q1/a

In [8]:
## Searching for .txt files

print(f"Searching in: {RAW_DATA_DIR}\n")

# Find all files ending in .txt / .TXT (case-insensitive)
all_txt_files = []
for root, dirs, files in os.walk(RAW_DATA_DIR):
    for f in files:
        if f.lower().endswith('.txt'):
            all_txt_files.append(os.path.join(root, f).replace("\\", "/"))

print(f"Total .txt files discovered: {len(all_txt_files)}")

# Group files by type

demo_files = []
drug_files = []
reac_files = []
indi_files = []
outc_files = []
rpsr_files = []
ther_files = []

# Scan every file without restricting extension conditions
for root, _, files in os.walk(PROJECT_ROOT):
    for f in files:
        if f.startswith('.'):
            continue
        
        full_path = os.path.join(root, f).replace("\\", "/")
        f_upper = f.upper()
        
        # Match filenames containing the table names
        if "DEMO" in f_upper and (f_upper.endswith(".TXT") or "." not in f):
            demo_files.append(full_path)
        elif "DRUG" in f_upper and (f_upper.endswith(".TXT") or "." not in f):
            drug_files.append(full_path)
        elif "REAC" in f_upper and (f_upper.endswith(".TXT") or "." not in f):
            reac_files.append(full_path)
        elif "INDI" in f_upper and (f_upper.endswith(".TXT") or "." not in f):
            indi_files.append(full_path)
        elif "OUTC" in f_upper and (f_upper.endswith(".TXT") or "." not in f):
            outc_files.append(full_path)
        elif "RPSR" in f_upper and (f_upper.endswith(".TXT") or "." not in f):
            rpsr_files.append(full_path)
        elif "THER" in f_upper and (f_upper.endswith(".TXT") or "." not in f):
            ther_files.append(full_path)

print(f"Total DEMO files found: {len(demo_files)}")
print(f"Total DRUG files found: {len(drug_files)}")
print(f"Total REAC files found: {len(reac_files)}")
print(f"Total INDI files found: {len(indi_files)}")
print(f"Total OUTC files found: {len(outc_files)}")
print(f"Total RPSR files found: {len(rpsr_files)}")
print(f"Total THER files found: {len(ther_files)}\n")

if demo_files:
    print(f"Sample DEMO path: {demo_files[-1]}")
if drug_files:
    print(f"Sample DRUG path: {drug_files[-1]}")
if reac_files:
    print(f"Sample DRUG path: {reac_files[-1]}")
if indi_files:
    print(f"Sample DRUG path: {indi_files[-1]}")
if outc_files:
    print(f"Sample DRUG path: {outc_files[-1]}")
if rpsr_files:
    print(f"Sample DRUG path: {rpsr_files[-1]}")
if ther_files:
    print(f"Sample REAC path: {ther_files[-1]}\n")

# Connect to DuckDB and inspect schemas
con = duckdb.connect()

if demo_files:
    print("=" * 60)
    print("📋 DEMO FILE SCHEMA & COLUMNS")
    print("=" * 60)
    display(con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{demo_files[0]}', delim='$', header=True, ignore_errors=True);").df()[['column_name', 'column_type']])

if drug_files:
    print("\n" + "=" * 60)
    print("📋 DRUG FILE SCHEMA & COLUMNS")
    print("=" * 60)
    display(con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{drug_files[0]}', delim='$', header=True, ignore_errors=True);").df()[['column_name', 'column_type']])

if reac_files:
    print("\n" + "=" * 60)
    print("📋 REAC FILE SCHEMA & COLUMNS")
    print("=" * 60)
    display(con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{reac_files[0]}', delim='$', header=True, ignore_errors=True);").df()[['column_name', 'column_type']])

if indi_files:
    print("\n" + "=" * 60)
    print("📋 INDI FILE SCHEMA & COLUMNS")
    print("=" * 60)
    display(con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{indi_files[0]}', delim='$', header=True, ignore_errors=True);").df()[['column_name', 'column_type']])

if outc_files:
    print("\n" + "=" * 60)
    print("📋 OUTC FILE SCHEMA & COLUMNS")
    print("=" * 60)
    display(con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{outc_files[0]}', delim='$', header=True, ignore_errors=True);").df()[['column_name', 'column_type']])

if rpsr_files:
    print("\n" + "=" * 60)
    print("📋 RPSR FILE SCHEMA & COLUMNS")
    print("=" * 60)
    display(con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{rpsr_files[0]}', delim='$', header=True, ignore_errors=True);").df()[['column_name', 'column_type']])

if ther_files:
    print("\n" + "=" * 60)
    print("📋 THER FILE SCHEMA & COLUMNS")
    print("=" * 60)
    display(con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{ther_files[0]}', delim='$', header=True, ignore_errors=True);").df()[['column_name', 'column_type']])



con.close()

Searching in: /Users/shymashameer/Desktop/Final Project/faers/data/raw

Total .txt files discovered: 173
Total DEMO files found: 24
Total DRUG files found: 24
Total REAC files found: 24
Total INDI files found: 24
Total OUTC files found: 24
Total RPSR files found: 24
Total THER files found: 24

Sample DEMO path: /Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2018q2/ascii/DEMO18Q2.txt
Sample DRUG path: /Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2018q2/ascii/DRUG18Q2.txt
Sample DRUG path: /Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2018q2/ascii/REAC18Q2.txt
Sample DRUG path: /Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2018q2/ascii/INDI18Q2.txt
Sample DRUG path: /Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2018q2/ascii/OUTC18Q2.txt
Sample DRUG path: /Users/shymashameer/Desktop/Final Project/faers/data/raw/faers_ascii_2018q2/ascii/RPSR18Q2.txt
Sample REAC path: /Users/sh

,column_name,column_type
0,primaryid,BIGINT
1,caseid,BIGINT
2,caseversion,BIGINT
3,i_f_code,BOOLEAN
4,event_dt,BIGINT
5,mfr_dt,BIGINT
6,init_fda_dt,BIGINT
7,fda_dt,BIGINT
8,rept_cod,VARCHAR
9,auth_num,VARCHAR



📋 DRUG FILE SCHEMA & COLUMNS


,column_name,column_type
0,primaryid,BIGINT
1,caseid,BIGINT
2,drug_seq,BIGINT
3,role_cod,VARCHAR
4,drugname,VARCHAR
5,prod_ai,VARCHAR
6,val_vbm,BIGINT
7,route,VARCHAR
8,dose_vbm,VARCHAR
9,cum_dose_chr,DOUBLE



📋 REAC FILE SCHEMA & COLUMNS


,column_name,column_type
0,primaryid,BIGINT
1,caseid,BIGINT
2,pt,VARCHAR
3,drug_rec_act,VARCHAR


... (4 outputs left out)


📋 RPSR FILE SCHEMA & COLUMNS


,column_name,column_type
0,primaryid,BIGINT
1,caseid,BIGINT
2,rpsr_cod,VARCHAR



📋 THER FILE SCHEMA & COLUMNS


,column_name,column_type
0,primaryid,BIGINT
1,caseid,BIGINT
2,dsg_drug_seq,BIGINT
3,start_dt,BIGINT
4,end_dt,BIGINT
5,dur,BIGINT
6,dur_cod,VARCHAR


## Selecting the Data for Analysis 

 Selecting Anti-Daibetic drugs from 2 clasess for this project (SGLT2 inhibitors & GLP-1 receptor agonists )

In [9]:

# Creating a complete build

# ============ CONFIG ============
DRUGS = {
    "dapagliflozin": ("SGLT2i", r"farxiga|forxiga|dapagliflozin"),
    "canagliflozin": ("SGLT2i", r"invokana|canagliflozin"),
    "empagliflozin": ("SGLT2i", r"jardiance|empagliflozin"),
    "ertugliflozin": ("SGLT2i", r"steglatro|ertugliflozin"),
    "liraglutide":   ("GLP1RA", r"victoza|saxenda|liraglutide"),
    "semaglutide":   ("GLP1RA", r"ozempic|semaglutide"),
    "dulaglutide":   ("GLP1RA", r"trulicity|dulaglutide"),
}
# Excluded from the primary exposure: combination products + out-of-scope brands
COMBO_RX = r"invokamet|synjardy|glyxambi|xigduo|segluromet|steglujan|qtern|soliqua|xultophy|rybelsus|wegovy"
COUNTRY, START, END = "US", "2014-01-01", "2019-12-31"

TXT = "lower(coalesce(drugname,'') || ' ' || coalesce(prod_ai,''))"
ING_SQL = "CASE " + " ".join(f"WHEN regexp_matches({TXT}, '{rx}') THEN '{i}'" for i,(_,rx) in DRUGS.items()) + " END"
CLS_SQL = "CASE " + " ".join(f"WHEN regexp_matches({TXT}, '{rx}') THEN '{c}'" for _,(c,rx) in DRUGS.items()) + " END"
ANY_RX  = "|".join(rx for _, rx in DRUGS.values()) + "|" + COMBO_RX      # built AFTER COMBO_RX

def collect_faers_files(prefix):
   matched_files = []
   for root, _, files in os.walk(RAW_DATA_DIR):
       for f in files:
           if f.startswith('.'):
               continue
           f_upper = f.upper()
           if prefix.upper() in f_upper and (f_upper.endswith(".TXT") or "." not in f):
               matched_files.append(os.path.join(root, f).replace("\\", "/"))
   return sorted(matched_files)


# ============ FILE LISTS (one variable per type) ============
demo_files = collect_faers_files("DEMO");  drug_files = collect_faers_files("DRUG")
reac_files = collect_faers_files("REAC");  outc_files = collect_faers_files("OUTC")
rpsr_files = collect_faers_files("RPSR");  indi_files = collect_faers_files("INDI")
ther_files = collect_faers_files("THER")

file_dict = dict(
    demo=demo_files, drug=drug_files, reac=reac_files,
    outc=outc_files, rpsr=rpsr_files, indi=indi_files, ther=ther_files
)

for name, files in file_dict.items():
    print(f"{name}: {len(files)}")

def read_faers(files):
   """One reader for every FAERS file type."""
   return f"""read_csv({files}, delim='$', header=True, quote='',
              all_varchar=True, union_by_name=True, ignore_errors=True)"""

# ==================== PIPELINE ==============================
def build_all(con):
    # 1. raw loads
        #1a. Demo rows
    con.execute(f"CREATE OR REPLACE TABLE raw_demo AS SELECT * FROM {read_faers(demo_files)}")
    print(f" 1a. DEMO rows: {con.execute('SELECT COUNT(*) FROM raw_demo').fetchone()[0]:,}")

        # 1b. Loading study-drug rows 
    con.execute(f"""CREATE OR REPLACE TABLE study_drug_rows AS
        SELECT primaryid, caseid, drug_seq, UPPER(role_cod) AS role_cod,
               UPPER(drugname) AS drugname, UPPER(prod_ai) AS prod_ai,
               UPPER(dechal) AS dechal, UPPER(rechal) AS rechal,
               route, dose_amt, dose_unit, dose_freq
        FROM {read_faers(drug_files)} WHERE regexp_matches({TXT}, '{ANY_RX}')""")
    print(f" 1b. Study-drug rows: {con.execute('SELECT COUNT(*) FROM study_drug_rows').fetchone()[0]:,}")

        # 1c. Loading REAC 
    con.execute(f"""CREATE OR REPLACE TABLE raw_reac AS
        SELECT primaryid, caseid, UPPER(pt) AS adverse_event FROM {read_faers(reac_files)}""")
    print(f" 1c. REAC rows: {con.execute('SELECT COUNT(*) FROM raw_reac').fetchone()[0]:,}")

        # 1d. Loading INDI 
    con.execute(f"""CREATE OR REPLACE TABLE raw_indi AS
        SELECT primaryid, indi_drug_seq AS drug_seq, upper(indi_pt) AS indi_pt FROM {read_faers(indi_files)}""")
    print(f" 1d. INDI rows: {con.execute('SELECT COUNT(*) FROM raw_indi').fetchone()[0]:,}")

        # 1e. Loading THER 
    con.execute(f"""CREATE OR REPLACE TABLE raw_ther AS
        SELECT primaryid, dsg_drug_seq AS drug_seq, start_dt, end_dt, dur, dur_cod FROM {read_faers(ther_files)}""")
    print(f" 1e. THER rows: {con.execute('SELECT COUNT(*) FROM raw_ther').fetchone()[0]:,}")



    # 2. Deduplicating DEMO one row per case (latest version of the case)
    con.execute("""CREATE OR REPLACE TABLE clean_demo AS
        SELECT * EXCLUDE (rnk) FROM (
          SELECT *, ROW_NUMBER() OVER (PARTITION BY caseid
                 ORDER BY TRY_CAST(fda_dt AS INTEGER) DESC NULLS LAST,
                          TRY_CAST(primaryid AS BIGINT) DESC) AS rnk
          FROM raw_demo) WHERE rnk = 1""")
    print(f" 2. Unique cases: {con.execute('SELECT COUNT(*) FROM clean_demo').fetchone()[0]:,}")

    # 3. analysis universe
    cty = "coalesce(nullif(occr_country,''), nullif(reporter_country,''))"
    cty_filter = f"AND {cty} = '{COUNTRY}'" if COUNTRY else ""
    con.execute(f"""CREATE OR REPLACE TABLE demo_us AS
        SELECT primaryid, caseid,
               TRY_STRPTIME(init_fda_dt,'%Y%m%d')::DATE AS init_dt,
               date_trunc('month', TRY_STRPTIME(init_fda_dt,'%Y%m%d'))::DATE AS init_month,
               TRY_STRPTIME(event_dt,'%Y%m%d')::DATE AS event_dt_parsed,
               TRY_CAST(age AS DOUBLE) AS age_raw, age_cod, sex, occp_cod, rept_cod,
               {cty} AS country
        FROM clean_demo
        WHERE TRY_STRPTIME(init_fda_dt,'%Y%m%d') BETWEEN DATE '{START}' AND DATE '{END}' {cty_filter}""")
    print(f" 3. Cases in window: {con.execute('SELECT COUNT(*) FROM demo_us').fetchone()[0]:,}")


    # 4. exposure and events
    con.execute(f"""CREATE OR REPLACE TABLE case_drug AS
        SELECT DISTINCT caseid, role_cod, ingredient, drug_class, is_combo FROM (
          SELECT u.caseid, s.role_cod, {ING_SQL} AS ingredient, {CLS_SQL} AS drug_class,
                 (regexp_matches({TXT}, '{COMBO_RX}') OR contains(coalesce(prod_ai,''), chr(92))) AS is_combo
          FROM study_drug_rows s JOIN demo_us u ON s.primaryid = u.primaryid)
        WHERE ingredient IS NOT NULL""")
    con.execute("""CREATE OR REPLACE TABLE case_pt AS
        SELECT DISTINCT u.caseid, u.init_month, r.adverse_event AS pt
        FROM raw_reac r JOIN demo_us u ON r.primaryid = u.primaryid
        WHERE r.adverse_event IS NOT NULL""")

    # 5. outcomes and sources
    con.execute(f"""CREATE OR REPLACE TABLE case_outcome AS
        SELECT u.caseid,
          bool_or(upper(o.outc_cod)='DE') AS death,  bool_or(upper(o.outc_cod)='LT') AS life_threat,
          bool_or(upper(o.outc_cod)='HO') AS hospitalised, bool_or(upper(o.outc_cod)='DS') AS disability,
          bool_or(upper(o.outc_cod)='CA') AS congenital, bool_or(upper(o.outc_cod)='RI') AS intervention,
          bool_or(upper(o.outc_cod)='OT') AS other_serious
        FROM {read_faers(outc_files)} o JOIN demo_us u ON o.primaryid = u.primaryid GROUP BY u.caseid""")
    con.execute(f"""CREATE OR REPLACE TABLE case_source AS
        SELECT u.caseid,
          bool_or(upper(r.rpsr_cod)='LIT') AS src_literature,
          bool_or(upper(r.rpsr_cod)='HP')  AS src_health_prof
        FROM {read_faers(rpsr_files)} r JOIN demo_us u ON r.primaryid = u.primaryid GROUP BY u.caseid""")

    # 6. case_base (every US case)
    con.execute("""CREATE OR REPLACE TABLE case_base AS
        SELECT s.caseid, s.init_dt, s.init_month, year(s.init_month) AS init_year,
               CASE WHEN s.age_years BETWEEN 0 AND 120 THEN s.age_years END AS age_years,
               s.sex, s.occp_cod, s.rept_cod, s.event_dt_parsed, s.country,
               (o.caseid IS NOT NULL) AS has_outcome,
               coalesce(o.death,false) AS death, coalesce(o.life_threat,false) AS life_threat,
               coalesce(o.hospitalised,false) AS hospitalised, coalesce(o.disability,false) AS disability,
               coalesce(o.congenital,false) AS congenital, coalesce(o.intervention,false) AS intervention,
               coalesce(o.other_serious,false) AS other_serious,
               (coalesce(o.death,false) OR coalesce(o.life_threat,false) OR coalesce(o.hospitalised,false)
                OR coalesce(o.disability,false) OR coalesce(o.congenital,false)) AS serious_core,
               coalesce(r.src_literature,false) AS src_literature,
               coalesce(r.src_health_prof,false) AS src_health_prof
        FROM (SELECT *, CASE upper(age_cod) WHEN 'YR' THEN age_raw WHEN 'DEC' THEN age_raw*10
                WHEN 'MON' THEN age_raw/12 WHEN 'WK' THEN age_raw/52.18
                WHEN 'DY' THEN age_raw/365.25 WHEN 'HR' THEN age_raw/8766 END AS age_years
              FROM demo_us) s
        LEFT JOIN case_outcome o USING (caseid) LEFT JOIN case_source r USING (caseid)""")

    # 7. drug-level detail (brand, indication, start date)
    con.execute(f"""CREATE OR REPLACE TABLE drug_detail AS
    SELECT u.caseid, s.drug_seq, s.role_cod, s.drugname, s.prod_ai, s.dechal, s.rechal,
           (regexp_matches(lower(coalesce(s.drugname,'') || ' ' || coalesce(s.prod_ai,'')), '{COMBO_RX}')
            OR contains(coalesce(s.prod_ai,''), chr(92))) AS is_combo,
           CASE WHEN s.drugname LIKE '%SAXENDA%' THEN 'SAXENDA' WHEN s.drugname LIKE '%VICTOZA%' THEN 'VICTOZA'
                WHEN s.drugname LIKE '%FORXIGA%' OR s.drugname LIKE '%FARXIGA%' THEN 'FARXIGA'
                WHEN s.drugname LIKE '%INVOKANA%' THEN 'INVOKANA' WHEN s.drugname LIKE '%JARDIANCE%' THEN 'JARDIANCE'
                WHEN s.drugname LIKE '%STEGLATRO%' THEN 'STEGLATRO' WHEN s.drugname LIKE '%OZEMPIC%' THEN 'OZEMPIC'
                WHEN s.drugname LIKE '%TRULICITY%' THEN 'TRULICITY' ELSE 'GENERIC/OTHER' END AS brand,
           i.indications, try_strptime(t.start_dt,'%Y%m%d')::DATE AS drug_start
    FROM study_drug_rows s JOIN demo_us u ON s.primaryid = u.primaryid
    LEFT JOIN (SELECT primaryid, drug_seq, list(DISTINCT indi_pt) AS indications
               FROM raw_indi GROUP BY 1,2) i ON i.primaryid=s.primaryid AND i.drug_seq=s.drug_seq
    LEFT JOIN (SELECT primaryid, drug_seq, min(start_dt) AS start_dt
               FROM raw_ther WHERE length(start_dt)=8 GROUP BY 1,2) t
           ON t.primaryid=s.primaryid AND t.drug_seq=s.drug_seq""")

    # 8. exposure groups and pair counts
    con.execute("""CREATE OR REPLACE TABLE exposure AS
        SELECT caseid, ingredient,
               CASE WHEN bool_or(role_cod='PS' AND NOT is_combo) THEN 'exp' ELSE 'excl' END AS g
        FROM case_drug GROUP BY 1,2""")
    con.execute("""CREATE OR REPLACE TABLE pair_counts AS
        WITH N AS (SELECT COUNT(DISTINCT caseid) AS n FROM case_pt),
        ev_tot AS (SELECT pt, COUNT(*) AS n_e FROM case_pt GROUP BY pt),
        grp_n  AS (SELECT ingredient, g, COUNT(*) AS n FROM exposure GROUP BY 1,2),
        ev_grp AS (SELECT x.ingredient, x.g, p.pt, COUNT(*) AS n
                   FROM exposure x JOIN case_pt p USING (caseid) GROUP BY 1,2,3)
        SELECT e.ingredient, e.pt, e.n AS a, ne.n AS n_exp, COALESCE(nx.n,0) AS n_excl,
               (SELECT n FROM N) AS N, t.n_e AS n_event_all, COALESCE(xe.n,0) AS n_event_excl
        FROM ev_grp e JOIN grp_n ne ON ne.ingredient=e.ingredient AND ne.g='exp'
        LEFT JOIN grp_n nx ON nx.ingredient=e.ingredient AND nx.g='excl'
        JOIN ev_tot t ON t.pt=e.pt
        LEFT JOIN ev_grp xe ON xe.ingredient=e.ingredient AND xe.pt=e.pt AND xe.g='excl'
        WHERE e.g='exp'""")

    # 9. wide study tables (no joins needed afterwards)
    ps  = ",\n".join(f"bool_or(role_cod='PS' AND NOT is_combo AND ingredient='{i}') AS ps_{i}" for i in DRUGS)
    anyc = ",\n".join(f"bool_or(ingredient='{i}') AS any_{i}" for i in DRUGS)
    con.execute(f"""CREATE OR REPLACE TABLE case_exposure AS
        SELECT caseid, {ps}, {anyc},
          bool_or(role_cod='PS' AND NOT is_combo AND drug_class='SGLT2i') AS ps_class_sglt2i,
          bool_or(role_cod='PS' AND NOT is_combo AND drug_class='GLP1RA') AS ps_class_glp1ra,
          bool_or(role_cod IN ('PS','SS') AND NOT is_combo AND drug_class='SGLT2i') AS use_sglt2i,
          bool_or(role_cod IN ('PS','SS') AND NOT is_combo AND drug_class='GLP1RA') AS use_glp1ra,
          bool_or(is_combo) AS has_combo,
          list(DISTINCT ingredient) FILTER (WHERE role_cod='PS' AND NOT is_combo) AS ps_ingredients
        FROM case_drug GROUP BY caseid""")
    con.execute("""CREATE OR REPLACE TABLE study_cases AS
        SELECT b.*, x.* EXCLUDE (caseid), (x.use_sglt2i AND x.use_glp1ra) AS dual_class,
               len(x.ps_ingredients) AS n_ps_ingredients, e.n_events, e.pts
        FROM case_exposure x JOIN case_base b USING (caseid)
        LEFT JOIN (SELECT caseid, COUNT(*) AS n_events, list(DISTINCT pt) AS pts
                   FROM case_pt GROUP BY caseid) e USING (caseid)""")
    con.execute("""CREATE OR REPLACE TABLE study_case_event AS
        SELECT p.caseid, p.pt, s.* EXCLUDE (caseid, init_month, pts, n_events), p.init_month
        FROM case_pt p JOIN study_cases s USING (caseid)""")

con = duckdb.connect(str(DB_PATH))
build_all(con)

demo: 24
drug: 24
reac: 24
outc: 24
rpsr: 24
indi: 24
ther: 24
 1a. DEMO rows: 8,309,659
 1b. Study-drug rows: 142,585
 1c. REAC rows: 25,084,108
 1d. INDI rows: 21,142,410
 1e. THER rows: 11,735,170
 2. Unique cases: 7,443,175
 3. Cases in window: 5,374,222


In [10]:
# 1. Case-Drug Role & Combo Breakdown For the selected drugs
df_roles = con.sql("""
    SELECT
      COUNT(DISTINCT caseid)                                                        AS any_study_case,
      COUNT(DISTINCT caseid) FILTER (WHERE role_cod='PS')                           AS ps_any,
      COUNT(DISTINCT caseid) FILTER (WHERE role_cod='PS' AND NOT is_combo)          AS ps_non_combo,
      COUNT(DISTINCT caseid) FILTER (WHERE role_cod='PS' AND is_combo)              AS ps_combo,
      COUNT(DISTINCT caseid) FILTER (WHERE role_cod IN ('PS','SS') AND NOT is_combo) AS ps_or_ss
    FROM case_drug
""").df()

r = df_roles.iloc[0]
print("--- Case-Drug Breakdown ---")
print(f"• Total Unique Cases (Any Role):        {r['any_study_case']:,}")
print(f"• Primary Suspect (PS) - Total:         {r['ps_any']:,}")
print(f"• Primary Suspect (PS) - Non-Combo:     {r['ps_non_combo']:,}")
print(f"• Primary Suspect (PS) - Combo:         {r['ps_combo']:,}")
print(f"• PS or Secondary Suspect (Non-Combo):  {r['ps_or_ss']:,}\n")


# 2. Unique cases and event rows you will actually analyse
df_analysis = con.sql("""
    SELECT COUNT(*) AS cases, SUM(n_events) AS case_event_rows
    FROM study_cases WHERE ps_class_sglt2i OR ps_class_glp1ra
""").df()

a = df_analysis.iloc[0]
print("--- Final Analysis Cohort (SGLT2i or GLP-1RA) ---")
print(f"• Filtered Cases:    {a['cases']:,}")
print(f"• Total Event Rows:  {a['case_event_rows']:,}")

--- Case-Drug Breakdown ---
• Total Unique Cases (Any Role):        84,790
• Primary Suspect (PS) - Total:         70,488
• Primary Suspect (PS) - Non-Combo:     67,653
• Primary Suspect (PS) - Combo:         2,835
• PS or Secondary Suspect (Non-Combo):  70,570

--- Final Analysis Cohort (SGLT2i or GLP-1RA) ---
• Filtered Cases:    67,653.0
• Total Event Rows:  155,068.0


### Build required tables & views

In [11]:
# rebuilds every monthly table, the metrics, and the two tables the views depend on

def metrics(df, min_a=1):
    df = df.astype({c: float for c in ["a","n_exp","n_excl","n_event_all","n_event_excl","N"]})
    b = df.n_exp - df.a
    c = df.n_event_all - df.a - df.n_event_excl
    d = (df.N - df.n_exp - df.n_excl) - c
    df = df.assign(b=b, c=c, d=d)
    df = df[(df.a >= min_a) & (df.b > 0) & (df.c > 0) & (df.d > 0)].copy()
    a, b, c, d = df.a, df.b, df.c, df.d
    df["PRR"]    = (a/(a+b)) / (c/(c+d))
    df["chi2"]   = (a*d - b*c)**2 * (a+b+c+d) / ((a+b)*(c+d)*(a+c)*(b+d))
    df["ROR"]    = (a*d) / (b*c)
    df["ROR_lo"] = np.exp(np.log(df.ROR) - 1.96*np.sqrt(1/a + 1/b + 1/c + 1/d))
    e            = (a+b)*(a+c)/(a+b+c+d)
    df["IC"]     = np.log2((a+0.5)/(e+0.5))
    df["IC025"]  = df.IC - 3.3*(a+0.5)**-0.5 - 2*(a+0.5)**-1.5
    df["signal"] = (a >= 3) & (df.PRR >= 2) & (df.chi2 >= 4) & (df.ROR_lo > 1) & (df.IC025 > 0)
    return df

def build_monthly(con, burn_in_end="2014-12-01"):
    con.execute(f"""CREATE OR REPLACE TABLE months AS
        SELECT unnest(generate_series(DATE '{START}', DATE '{END}', INTERVAL 1 MONTH))::DATE AS m""")
    con.execute("CREATE OR REPLACE TABLE case_month AS SELECT DISTINCT caseid, init_month AS m FROM case_pt")
    con.execute("CREATE OR REPLACE TABLE m_N AS SELECT m, COUNT(*) AS n FROM case_month GROUP BY m")
    con.execute("""CREATE OR REPLACE TABLE m_grp AS
        SELECT x.ingredient, x.g, c.m, COUNT(*) AS n
        FROM exposure x JOIN case_month c USING (caseid) GROUP BY ALL""")
    con.execute("CREATE OR REPLACE TABLE m_ev AS SELECT pt, init_month AS m, COUNT(*) AS n FROM case_pt GROUP BY ALL")
    con.execute("""CREATE OR REPLACE TABLE m_pair AS
        SELECT x.ingredient, x.g, p.pt, p.init_month AS m, COUNT(*) AS n
        FROM exposure x JOIN case_pt p USING (caseid) GROUP BY ALL""")

    con.execute("""
    CREATE OR REPLACE TABLE pair_month AS
    WITH pairs AS (SELECT ingredient, pt FROM pair_counts WHERE a >= 3),
    grid AS (SELECT p.ingredient, p.pt, mo.m FROM pairs p CROSS JOIN months mo),
    j AS (
      SELECT g.ingredient, g.pt, g.m,
             COALESCE(a.n,0) AS a_m, COALESCE(xe.n,0) AS xe_m, COALESCE(ev.n,0) AS ev_m,
             COALESCE(ge.n,0) AS exp_m, COALESCE(gx.n,0) AS excl_m, COALESCE(nn.n,0) AS N_m
      FROM grid g
      LEFT JOIN m_pair a  ON a.ingredient=g.ingredient  AND a.pt=g.pt  AND a.m=g.m  AND a.g='exp'
      LEFT JOIN m_pair xe ON xe.ingredient=g.ingredient AND xe.pt=g.pt AND xe.m=g.m AND xe.g='excl'
      LEFT JOIN m_ev ev   ON ev.pt=g.pt AND ev.m=g.m
      LEFT JOIN m_grp ge  ON ge.ingredient=g.ingredient AND ge.m=g.m AND ge.g='exp'
      LEFT JOIN m_grp gx  ON gx.ingredient=g.ingredient AND gx.m=g.m AND gx.g='excl'
      LEFT JOIN m_N nn    ON nn.m=g.m)
    SELECT ingredient, pt, m,
           CAST(SUM(a_m)    OVER w AS BIGINT) AS a,
           CAST(SUM(exp_m)  OVER w AS BIGINT) AS n_exp,
           CAST(SUM(excl_m) OVER w AS BIGINT) AS n_excl,
           CAST(SUM(ev_m)   OVER w AS BIGINT) AS n_event_all,
           CAST(SUM(xe_m)   OVER w AS BIGINT) AS n_event_excl,
           CAST(SUM(N_m)    OVER w AS BIGINT) AS N
    FROM j
    WINDOW w AS (PARTITION BY ingredient, pt ORDER BY m)
    """)

    pm = con.sql("SELECT * FROM pair_month").df()
    ts = metrics(pm)
    ts["m"] = pd.to_datetime(ts["m"])

    keys = ["ingredient", "pt"]
    onset = ts[ts.signal].groupby(keys).m.min().rename("onset")
    already = ts[(ts.m == pd.Timestamp(burn_in_end)) & ts.signal].set_index(keys).index
    labels = pm[keys].drop_duplicates().set_index(keys).join(onset)
    labels["status"] = np.where(labels.onset.isna(), "never",
                       np.where(labels.index.isin(already), "pre-existing", "emerging"))

    con.register("ts_df", ts)
    con.register("labels_df", labels.reset_index())
    con.execute("CREATE OR REPLACE TABLE signal_ts AS SELECT * FROM ts_df")
    con.execute("CREATE OR REPLACE TABLE signal_labels AS SELECT * FROM labels_df")
    
    con.unregister("labels_df")
    con.unregister("ts_df")
    return ts, labels


In [12]:
# monthly tables, metrics, signal_ts, signal_labels
ts, labels = build_monthly(con)

In [13]:
# To check if signal_ts & signal_labels were created correctly

print(callable(metrics), callable(build_monthly), START, END)
tables = con.sql("SHOW TABLES").df().name.tolist()
for t in ["signal_ts", "signal_labels", "pair_month", "m_pair", "months"]:
    print(f"{t:15s}", "EXISTS" if t in tables else "MISSING")

True True 2014-01-01 2019-12-31
signal_ts       EXISTS
signal_labels   EXISTS
pair_month      EXISTS
m_pair          EXISTS
months          EXISTS


In [14]:
# Creating a build_layer for reference tables, ps_* tables, views

def build_layer(con):
    # ---------- reference tables ----------
    ref_drug = pd.DataFrame([(i, c) for i, (c, _) in DRUGS.items()],
                            columns=["ingredient", "drug_class"])
    con.register("ref_drug_df", ref_drug)
    con.execute("CREATE OR REPLACE TABLE ref_drug AS SELECT * FROM ref_drug_df")
    con.unregister("ref_drug_df")
    
    known = pd.DataFrame([
        ("dapagliflozin", "KETOACIDOSIS",         "2015-05-15", "FDA DSC: SGLT2i and ketoacidosis"),
        ("empagliflozin", "KETOACIDOSIS",         "2015-05-15", "FDA DSC: SGLT2i and ketoacidosis"),
        ("canagliflozin", "KETOACIDOSIS",         "2015-05-15", "FDA DSC: SGLT2i and ketoacidosis"),
        ("canagliflozin", "AMPUTATION",           "2016-05-18", "FDA DSC: interim CANVAS amputation risk"),
        ("dapagliflozin", "FOURNIER",             "2018-08-29", "FDA DSC: SGLT2i and Fournier's gangrene"),
        ("empagliflozin", "FOURNIER",             "2018-08-29", "FDA DSC: SGLT2i and Fournier's gangrene"),
        ("canagliflozin", "FOURNIER",             "2018-08-29", "FDA DSC: SGLT2i and Fournier's gangrene"),
        ("liraglutide",   "PANCREATITIS",         None,         "On label from approval"),
        ("dulaglutide",   "PANCREATITIS",         None,         "On label from approval"),
        ("semaglutide",   "PANCREATITIS",         None,         "On label from approval"),
    ], columns=["ingredient", "pt_regex", "reg_date", "source"])
    known["reg_date"] = pd.to_datetime(known["reg_date"])
    con.register("known_df", known)
    con.execute("CREATE OR REPLACE TABLE ref_known_signal AS SELECT * FROM known_df")
    con.unregister("known_df")

    # ---------- case-level tables ----------
    con.execute("""
    CREATE OR REPLACE TABLE ps_case AS
    SELECT x.ingredient, r.drug_class, s.*
    FROM exposure x
    JOIN ref_drug r USING (ingredient)
    JOIN study_cases s USING (caseid)
    WHERE x.g = 'exp'
    """)

    con.execute("""
    CREATE OR REPLACE TABLE ps_case_event AS
    SELECT c.ingredient, c.drug_class, p.pt, c.caseid, c.init_month, c.init_year,
           c.age_years, c.sex, c.occp_cod, c.rept_cod, c.event_dt_parsed,
           c.has_outcome, c.serious_core, c.death, c.life_threat, c.hospitalised,
           c.disability, c.src_literature, c.src_health_prof,
           c.dual_class, c.has_combo
    FROM ps_case c JOIN case_pt p USING (caseid)
    """)

    # ---------- drug-level table (brand, indication, onset) ----------
    con.execute(f"""
    CREATE OR REPLACE TABLE ps_drug AS
    SELECT {ING_SQL} AS ingredient, d.caseid, d.drug_seq, d.brand, d.drugname, d.prod_ai,
           d.dechal, d.rechal, d.indications, d.drug_start,
           b.init_month, b.event_dt_parsed,
           date_diff('day', d.drug_start, b.event_dt_parsed) AS days_to_onset,
           CASE WHEN d.indications IS NULL THEN 'unknown'
                WHEN regexp_matches(array_to_string(d.indications, ';'), 'OBESITY|WEIGHT')
                     THEN 'obesity/weight'
                ELSE 'diabetes/other' END AS indication_group
    FROM drug_detail d JOIN case_base b USING (caseid)
    WHERE d.role_cod = 'PS' AND NOT d.is_combo AND {ING_SQL} IS NOT NULL
    """)

    # ---------- views ----------
    con.execute("""
    CREATE OR REPLACE VIEW v_pair_stats AS
    SELECT *, (prr >= 2 AND chi2 >= 4 AND ror_lo > 1 AND ic025 > 0) AS signal
    FROM (
      SELECT ingredient, pt, a::INT AS a, b::BIGINT AS b, c::BIGINT AS c, d::BIGINT AS d,
             prr, chi2, ror, exp(ln(ror) - 1.96*se) AS ror_lo,
             log2((a+0.5)/(e+0.5)) AS ic,
             log2((a+0.5)/(e+0.5)) - 3.3*power(a+0.5,-0.5) - 2*power(a+0.5,-1.5) AS ic025
      FROM (
        SELECT ingredient, pt, a, b, c, d,
               (a/(a+b)) / (c/(c+d))                                   AS prr,
               (a*d) / (b*c)                                           AS ror,
               sqrt(1/a + 1/b + 1/c + 1/d)                             AS se,
               (a+b)*(a+c)/(a+b+c+d)                                   AS e,
               (a*d-b*c)*(a*d-b*c)*(a+b+c+d)/((a+b)*(c+d)*(a+c)*(b+d)) AS chi2
        FROM (
          SELECT ingredient, pt,
                 a::DOUBLE AS a,
                 (n_exp - a)::DOUBLE AS b,
                 (n_event_all - a - n_event_excl)::DOUBLE AS c,
                 ((N - n_exp - n_excl) - (n_event_all - a - n_event_excl))::DOUBLE AS d
          FROM pair_counts)
        WHERE a >= 3 AND b > 0 AND c > 0 AND d > 0))
    """)

    con.execute("""
    CREATE OR REPLACE VIEW v_pair_serious AS
    SELECT ingredient, pt, COUNT(*) AS cases,
           COUNT(*) FILTER (WHERE serious_core)  AS serious_cases,
           COUNT(*) FILTER (WHERE death)         AS deaths,
           COUNT(*) FILTER (WHERE hospitalised)  AS hospitalised,
           COUNT(*) FILTER (WHERE life_threat)   AS life_threat
    FROM ps_case_event GROUP BY ingredient, pt
    """)

    con.execute("""
    CREATE OR REPLACE VIEW v_signals AS
    SELECT s.*, l.onset, l.status,
           v.serious_cases, v.deaths, v.hospitalised,
           round(100.0 * v.serious_cases / s.a, 1) AS pct_serious
    FROM v_pair_stats s
    LEFT JOIN signal_labels l USING (ingredient, pt)
    LEFT JOIN v_pair_serious v USING (ingredient, pt)
    WHERE s.signal
    """)

    con.execute("""
    CREATE OR REPLACE VIEW v_known_signal_check AS
       WITH m AS (
    SELECT k.ingredient, k.pt_regex, k.reg_date, k.source,
         s.pt, s.a, s.ic025, s.signal, l.onset, l.status,
         (l.onset + INTERVAL 1 MONTH - INTERVAL 1 DAY)::DATE AS detect_date
    FROM ref_known_signal k
    JOIN v_pair_stats s ON s.ingredient = k.ingredient AND regexp_matches(s.pt, k.pt_regex)
    LEFT JOIN signal_labels l ON l.ingredient = s.ingredient AND l.pt = s.pt)
    SELECT *, date_diff('day', detect_date, reg_date) AS lead_days FROM m
""")

    con.execute("""
    CREATE OR REPLACE VIEW v_known_signal_family AS
    SELECT ingredient, pt_regex, reg_date,
       arg_min(pt, detect_date)     AS first_pt,
       arg_min(status, detect_date) AS status,
       min(detect_date)             AS first_detect,
       date_diff('day', min(detect_date), reg_date) AS lead_days
    FROM v_known_signal_check
    WHERE detect_date IS NOT NULL
    GROUP BY ingredient, pt_regex, reg_date
""")
    print(con.sql("SELECT * FROM v_known_signal_family ORDER BY ingredient, pt_regex").df())

    con.execute("""
    CREATE OR REPLACE VIEW v_monthly_cases AS
    SELECT ingredient, init_month AS m, COUNT(*) AS cases,
           COUNT(*) FILTER (WHERE serious_core) AS serious_cases
    FROM ps_case GROUP BY ingredient, init_month
    """)

    con.execute("""
    CREATE OR REPLACE VIEW v_pair_monthly AS
    SELECT ingredient, pt, m, n AS new_cases FROM m_pair WHERE g = 'exp'
    """)

    con.execute("""
    CREATE OR REPLACE VIEW v_ingredient_summary AS
    SELECT ingredient, any_value(drug_class) AS drug_class,
           COUNT(*) AS ps_cases,
           min(init_month) AS first_month, max(init_month) AS last_month,
           round(100*avg(serious_core::INT),1) AS pct_serious,
           round(100*avg(death::INT),1) AS pct_death,
           round(100*avg((sex='F')::INT),1) AS pct_female,
           round(median(age_years),1) AS median_age,
           round(100*avg(dual_class::INT),1) AS pct_dual_class,
           round(100*avg(src_literature::INT),1) AS pct_literature
    FROM ps_case GROUP BY ingredient
    """)

build_layer(con)
print(con.sql("""SELECT table_name, table_type FROM information_schema.tables
                 WHERE table_schema='main' ORDER BY table_type, table_name""").df())





      ingredient      pt_regex   reg_date               first_pt  \
0  canagliflozin    AMPUTATION 2016-05-18         TOE AMPUTATION   
1  canagliflozin      FOURNIER 2018-08-29    FOURNIER'S GANGRENE   
2  canagliflozin  KETOACIDOSIS 2015-05-15  DIABETIC KETOACIDOSIS   
3  dapagliflozin      FOURNIER 2018-08-29    FOURNIER'S GANGRENE   
4  dapagliflozin  KETOACIDOSIS 2015-05-15  DIABETIC KETOACIDOSIS   
5    dulaglutide  PANCREATITIS        NaT           PANCREATITIS   
6  empagliflozin      FOURNIER 2018-08-29    FOURNIER'S GANGRENE   
7  empagliflozin  KETOACIDOSIS 2015-05-15  DIABETIC KETOACIDOSIS   
8    liraglutide  PANCREATITIS        NaT           PANCREATITIS   
9    semaglutide  PANCREATITIS        NaT           PANCREATITIS   

         status first_detect  lead_days  
0      emerging   2016-06-30        -43  
1      emerging   2018-11-30        -93  
2  pre-existing   2014-02-28        441  
3      emerging   2018-12-31       -124  
4  pre-existing   2014-09-30        227  

In [15]:
## Creating a comprehensive table to help with exploration
con.execute("""
CREATE OR REPLACE TABLE faers_flat AS
WITH d AS (   -- one row per case x ingredient
  SELECT caseid, ingredient,
         string_agg(DISTINCT brand, '/')                           AS brand,
         CASE WHEN bool_or(indication_group = 'obesity/weight')  THEN 'obesity/weight'
              WHEN bool_or(indication_group = 'diabetes/other')  THEN 'diabetes/other'
              ELSE 'unknown' END                                   AS indication_group,
         string_agg(DISTINCT array_to_string(indications, '; '), ' | ') AS indications,
         min(drug_start)                                           AS drug_start,
         min(days_to_onset) FILTER (WHERE days_to_onset >= 0)      AS days_to_onset,
         string_agg(DISTINCT dechal, '/')                          AS dechal,
         string_agg(DISTINCT rechal, '/')                          AS rechal
  FROM ps_drug GROUP BY caseid, ingredient)
SELECT e.caseid, u.primaryid,
       e.init_month, e.init_year, b.country,
       e.age_years, e.sex, e.occp_cod, e.rept_cod,
       e.ingredient, e.drug_class,
       d.brand, d.indication_group, d.indications,
       d.drug_start, e.event_dt_parsed AS event_dt, d.days_to_onset, d.dechal, d.rechal,
       e.pt AS event,
       e.has_outcome, e.serious_core, e.death, e.life_threat, e.hospitalised, e.disability,
       b.congenital, b.intervention, b.other_serious,
       e.src_literature, e.src_health_prof, e.dual_class
FROM ps_case_event e
JOIN case_base b USING (caseid)
JOIN demo_us u   USING (caseid)
LEFT JOIN d ON d.caseid = e.caseid AND d.ingredient = e.ingredient
""")

print(con.sql("""SELECT (SELECT COUNT(*) FROM faers_flat)     AS flat_rows,
                        (SELECT COUNT(*) FROM ps_case_event) AS ps_case_event_rows,
                        COUNT(DISTINCT caseid)               AS cases FROM faers_flat""").df())

   flat_rows  ps_case_event_rows  cases
0     155068              155068  67653


In [ ]:
# pair_grid tabel to help with disproportionality analysis
con.execute("""
CREATE OR REPLACE TABLE pair_grid AS
WITH ev_tot AS (SELECT pt, COUNT(*) AS n_e FROM case_pt GROUP BY pt),
grp_n  AS (SELECT ingredient, g, COUNT(*) AS n FROM exposure GROUP BY 1, 2),
ev_grp AS (SELECT x.ingredient, x.g, p.pt, COUNT(*) AS n
           FROM exposure x JOIN case_pt p USING (caseid) GROUP BY 1, 2, 3),
ings AS (SELECT ingredient FROM grp_n WHERE g = 'exp')
SELECT i.ingredient, t.pt,
       COALESCE(a.n, 0)  AS a,
       ne.n              AS n_exp,
       COALESCE(nx.n, 0) AS n_excl,
       (SELECT COUNT(DISTINCT caseid) FROM case_pt) AS N,
       t.n_e             AS n_event_all,
       COALESCE(xe.n, 0) AS n_event_excl
FROM ings i CROSS JOIN ev_tot t
JOIN grp_n ne ON ne.ingredient = i.ingredient AND ne.g = 'exp'
LEFT JOIN grp_n nx ON nx.ingredient = i.ingredient AND nx.g = 'excl'
LEFT JOIN ev_grp a  ON a.ingredient  = i.ingredient AND a.pt  = t.pt AND a.g  = 'exp'
LEFT JOIN ev_grp xe ON xe.ingredient = i.ingredient AND xe.pt = t.pt AND xe.g = 'excl'
""")
print(con.sql("SELECT COUNT(*) cells, SUM((a>0)::INT) observed FROM pair_grid").df())

In [16]:
con.close()

# Explicitly close any ambient DuckDB connections
try:
    duckdb.close()
    print("✓ Closed ambient DuckDB connection.")
except Exception as e:
    print(f"Notice: {e}")

✓ Closed ambient DuckDB connection.
